# v3 — Hill Climbing: Search Visualization

How Hill Climbing moves through the search space on the synthetic FD
model: the objective of the current solution along the run, the number
of picks, snapshots of the solution over the semblance panel, where the
evaluations were spent, the restarts, and an animation.

Formulation and parameters: the same as the batch
`configs/batch_fd_v3_sep200.json` (v3, robust semblance, 200 ms
separation, 100,000 evaluations).

**Execution order:**
1. Setup
2. Parameters
3. Load the case
4. Run with the trajectory recorded
5. Objective and number of picks along the run
6. Snapshots of the current solution
7. Where the search looked
8. Restarts
9. Animation
10. Summary

## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Project root: walk up to the folder that contains src/
from pathlib import Path
project_root = Path.cwd().resolve()
while not (project_root / 'src').is_dir():
    if project_root.parent == project_root:
        raise RuntimeError('Project root (folder containing src/) not found.')
    project_root = project_root.parent
project_root = str(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')

In [ ]:
from src.seismic.cases      import load_case
from src.seismic.semblance  import compute_semblance, robust_threshold, ROBUST_SEMBLANCE
from src.analysis.reference import get_reference, match_reference
from src.analysis           import search_viz
from src.optimizers.hill_climbing import HillClimbing

print('All modules imported successfully.')

## 2. Parameters

In [ ]:
ALG = 'Hill Climbing'
TAG = 'hc'

# --- Case: synthetic FD model, with ('_fs') or without ('_nofs') free-surface multiples ---
CASE         = 'fd_layered_cdp2500_fs'

# --- Formulation: same as configs/batch_fd_v3_sep200.json (v3, robust semblance) ---
VEL_MIN      = 1400.0
VEL_MAX      = 5000.0
VEL_STEP     = 20.0          # panel used for tau and for the figures
MIN_SEP_MS   = 200.0
TAU_FACTOR   = 10.0
P_INSERT     = 0.1
P_REMOVE     = 0.1
SEMBLANCE    = ROBUST_SEMBLANCE   # stretch mute 0.5, energy floor 1%, min fold 10

# --- Budget and seed ---
MAX_EVALS    = 100000
SEED         = 42

# --- Local moves ---
STEP_TIME    = 5
STEP_VEL     = 50.0
N_NEIGHBORS  = 30
MAX_ITER     = 3 * MAX_EVALS // N_NEIGHBORS

# --- Hill Climbing ---
HC_PATIENCE  = 60            # iterations without improvement before a restart
HC_RESTARTS  = 10**6         # restart until the budget is used up

## 3. Load the case

Semblance panel and tau with the same options as the objective, and the
reference picks (semblance peak of each expected event) for the figures:
`+` white = primary, `+` magenta = surface multiple. The algorithm never
sees the reference.

In [ ]:
case = load_case(os.path.join(project_root, 'cases', f'{CASE}.npz'))
gather, offsets, dt = case['gather'], case['offsets'], case['dt_ms']
t_grid = case['t_grid']

velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)
panel  = compute_semblance(gather, offsets, velocities, dt, **SEMBLANCE)
TAU    = robust_threshold(panel, factor=TAU_FACTOR)
extent = [VEL_MIN, VEL_MAX, t_grid[-1], t_grid[0]]
reference = get_reference(case, **SEMBLANCE)

print(f'Case : {CASE}   gather {gather.shape}   tau = {TAU:.3f}')

## 4. Run with the trajectory recorded

`record_trajectory=True` stores the current solution at every iteration. It only reads the search state, so the run is identical to the one without recording (same seed, same result).

In [ ]:
opt = HillClimbing(
    gather, offsets, dt, vel_min=VEL_MIN, vel_max=VEL_MAX, min_sep_ms=MIN_SEP_MS,
    objective='threshold', tau=TAU, p_insert=P_INSERT, p_remove=P_REMOVE,
    seed=SEED, max_evals=MAX_EVALS, max_iter=MAX_ITER,
    step_time=STEP_TIME, step_vel=STEP_VEL, n_neighbors=N_NEIGHBORS,
    restarts=HC_RESTARTS, patience=HC_PATIENCE,
    record_trajectory=True, **SEMBLANCE,
).run()

print(f'f = {opt.best_score:.3f}   K = {len(opt.best_picks)}   evals = {opt.n_evals}   '
      f'time = {opt.exec_time_s:.1f} s')
print(f'Trajectory records: {len(opt.trajectory)}   new picks evaluated: {len(opt.proposals)}')

t_s, v_s = opt.get_result()
m = match_reference(t_s * dt, v_s, reference)
print(f"Best solution: on primary {m['n_on_primary']}/{m['n_primaries']}, "
      f"on multiple {m['n_on_multiple']}, recall {m['recall']:.0%}, F1 {m['f1']:.2f}")

## 5. Objective and number of picks along the run

Top: objective of the current solution (blue) and best so far (red);
gray lines are restarts. Hill Climbing only accepts improvements, so
inside one restart the blue curve only goes up — it climbs until no
neighbor is better, waits `HC_PATIENCE` iterations and restarts from a
new random solution (the blue curve drops to a low value). Bottom:
number of picks K of the current solution.

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'figures'), exist_ok=True)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
search_viz.plot_score_trace(opt, ax=axes[0])
axes[0].set_title(f'{ALG} — objective along the run ({CASE})')
search_viz.plot_n_picks(opt, ax=axes[1])
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', f'search_{TAG}_{CASE}_trace.png'), dpi=150)
plt.show()

## 6. Snapshots of the current solution

Six moments of the run, evenly spaced. White circles: current solution;
black X: best solution found up to that moment (when different).

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 7), sharey=True)
search_viz.plot_snapshots(opt, panel, extent, dt, n=6, reference=reference, axes=axes)
for ax in axes:
    ax.set_xlim(VEL_MIN, 3000)
plt.suptitle(f'{ALG} — current solution during the run ({CASE})', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', f'search_{TAG}_{CASE}_snapshots.png'),
            dpi=130, bbox_inches='tight')
plt.show()

## 7. Where the search looked

Every new pick the algorithm evaluated (a pick of an evaluated solution
that was not in the current one), as a 2D histogram over the panel (gray).
Dense regions are where the evaluations were spent; regions without
color were never tried.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
search_viz.plot_proposals(opt, panel, extent, dt, ax=ax, reference=reference)
ax.set_title(f'{ALG} — new picks evaluated ({len(opt.proposals)}) — {CASE}')
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', f'search_{TAG}_{CASE}_proposals.png'), dpi=150)
plt.show()

## 8. Restarts

Final objective of every restart, in order, and how long each one ran.
Most restarts end in a local optimum below the best; the best solution
comes from the few that happen to start in the right basin.

In [ ]:
scores = np.array(opt.restart_scores)
lengths = np.array([len(h) - 1 for h in opt.restart_histories])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(scores, '.', ms=4)
axes[0].axhline(opt.best_score, color='tab:red', lw=1, label=f'best = {opt.best_score:.2f}')
axes[0].set_xlabel('Restart'); axes[0].set_ylabel('Final f of the restart')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].hist(lengths, bins=30)
axes[1].set_xlabel('Iterations in the restart'); axes[1].set_ylabel('Number of restarts')
axes[1].grid(True, alpha=0.3)
plt.suptitle(f'Hill Climbing restarts ({CASE})')
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', f'search_{TAG}_{CASE}_restarts.png'), dpi=150)
plt.show()

near = np.mean(scores >= 0.99 * opt.best_score)
print(f'{len(scores)} restarts; {near:.0%} of them ended within 1% of the best; '
      f'median length {np.median(lengths):.0f} iterations')

## 9. Animation

GIF of the current solution moving over the panel (subsampled to at most
`MAX_FRAMES` frames). Saved in `results/figures/`. It can take a minute.

In [ ]:
MAX_FRAMES = 200
FPS        = 10
gif = os.path.join(project_root, 'results', 'figures', f'search_{TAG}_{CASE}.gif')
search_viz.animate_search(opt, panel, extent, dt, gif, max_frames=MAX_FRAMES, fps=FPS,
                          reference=reference, title=ALG)
print(f'Saved: {gif}')
display(Image(filename=gif))

## 10. Summary

**What to look for in Hill Climbing:**
- the objective of the current solution rises in steps inside each
  restart and then falls at every restart: the search climbs one basin
  at a time and never accepts a worse solution;
- the snapshots and the animation show each restart starting from a
  random set of picks (yellow frame in the GIF) and pulling them to the
  nearest blobs;
- the proposal density shows the evaluations spread over the whole panel
  (random starts) with concentrations around the blobs where restarts
  end — and how little of the budget lands on a small blob such as
  top C with the free surface;
- the restart statistics show how much of the budget is spent on
  restarts that end below the best.